In [1]:
from langgraph.graph import StateGraph,START,END
from pydantic import Field,BaseModel
from typing import TypedDict,Literal
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
model = ChatGroq(model="llama-3.3-70b-versatile")

In [4]:
class StructureSentiment(BaseModel):
    sentiment:Literal["positive","negative"]=Field(description="give me the sentiment of the Review as positive or negative")

In [5]:
structure_model=model.with_structured_output(StructureSentiment)

In [6]:
class ReviewState(TypedDict):
    review:str
    sentiment:str
    diagnose:dict
    reply:str

In [7]:
class Diagnose(BaseModel):
    issue_type:str=Field(description="Give me the issue type of the review")
    tone:str=Field(description="What is the tone of the review")
    urgency:str=Field(description="what is the tone of the review")

In [8]:
diagnose_model=model.with_structured_output(Diagnose)

In [9]:
graph=StateGraph(ReviewState)

In [10]:
def find_sentiment(state:ReviewState):
    prompt=f"Give me the sentiment of the review {state["review"]}"
    result=structure_model.invoke(prompt)
    return {"sentiment":result.sentiment}

def positive_reply(state:ReviewState):
    prompt=f"Give a proper reply to the postive sentiment review {state["review"]}"
    result=model.invoke(prompt)
    return {"reply":result.content}

def negative_diagnose(state:ReviewState):
    prompt=f"Give me the diagnose of the following review {state["review"]}"
    result=diagnose_model.invoke(prompt)
    return {"diagnose":result}
def negative_reply(state:ReviewState):
    prompt=f"Give a reply to the neagtive review which diagnosed as {state["diagnose"]}"
    result=model.invoke(prompt)
    return {"reply":result.content}

def condition(state:ReviewState)->Literal["negative_diagnose","positive_reply"]:
    if state["sentiment"]=="positive":
        return "positive_reply"
    else:
        return "negative_diagnose"

In [11]:
graph.add_node("find_sentiment",find_sentiment)
graph.add_node("positive_reply",positive_reply)
graph.add_node("negative_diagnose",negative_diagnose)
graph.add_node("negative_reply",negative_reply)

graph.add_edge(START,"find_sentiment")
graph.add_conditional_edges("find_sentiment",condition)
graph.add_edge("negative_diagnose","negative_reply")
graph.add_edge("negative_reply",END)
graph.add_edge("positive_reply",END)

In [12]:
workflow=graph.compile()

In [13]:
workflow.invoke({"review":"This app is incredibly fast, intuitive, and has never crashed once. Best purchase I've made this year!"})

{'review': "This app is incredibly fast, intuitive, and has never crashed once. Best purchase I've made this year!",
 'sentiment': 'positive',
 'reply': '"Thank you so much for taking the time to share your amazing experience with our app. We\'re thrilled to hear that you\'ve found it to be fast, intuitive, and reliable. We\'re proud of the work our team has put into creating a seamless user experience, and it\'s wonderful to know that it\'s made such a positive impact on your daily life. We\'re honored that you consider it the best purchase you\'ve made this year - that means the world to us! If you have any other feedback or suggestions, please don\'t hesitate to reach out. Thank you again for your kind words and for being a valued customer!"'}